In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import timm
from tqdm import tqdm
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score
import time

# ============================
# CONFIG
# ============================
DATA_DIR = r"C:\Users\m8494\Downloads\8020data"
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
VAL_DIR = os.path.join(DATA_DIR, 'val')
IMG_SIZE = 160
BATCH_SIZE = 16
EPOCHS = 100
LR = 1e-4
NUM_CLASSES = 3
MODEL_SAVE = 'efficientnet_b0_cpu.pth'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.set_num_threads(4)

# ============================
# TRANSFORMS
# ============================
train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop((IMG_SIZE, IMG_SIZE), scale=(0.8,1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.CenterCrop((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

# ============================
# DATALOADERS
# ============================
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_tfms)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=val_tfms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

class_names = train_dataset.classes
print("Classes:", class_names)
print("Train images:", len(train_dataset), "Val images:", len(val_dataset))
print("Device:", DEVICE)

# ============================
# MODEL
# ============================
model = timm.create_model('efficientnet_b0', pretrained=True)
in_features = model.classifier.in_features if hasattr(model, 'classifier') else model.get_classifier().in_features
model.classifier = nn.Linear(in_features, NUM_CLASSES)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

# ============================
# METRIC STORAGE
# ============================
train_loss_hist = []
val_loss_hist = []
train_acc_hist = []
val_acc_hist = []
prec_hist = []
recall_hist = []
f1_hist = []
epoch_time_hist = []

# ============================
# TRAINING LOOP
# ============================
best_val_acc = 0.0
total_start = time.time()

for epoch in range(1, EPOCHS+1):

    epoch_start = time.time()

    # ---------- TRAIN ----------
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_total = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [train]", leave=False)
    for images, labels in loop:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        running_total += labels.size(0)
        loop.set_postfix(loss=running_loss/running_total, acc=100*running_correct/running_total)

    train_loss = running_loss / running_total
    train_acc = running_correct / running_total

    # ---------- VALIDATION ----------
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [val]", leave=False):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)

            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_loss = val_loss / val_total
    val_acc = val_correct / val_total

    # METRICS
    precision = precision_score(all_labels, all_preds, average='macro')
    recall = recall_score(all_labels, all_preds, average='macro')
    f1 = f1_score(all_labels, all_preds, average='macro')

    epoch_time = time.time() - epoch_start

    # store metrics
    train_loss_hist.append(train_loss)
    val_loss_hist.append(val_loss)
    train_acc_hist.append(train_acc)
    val_acc_hist.append(val_acc)
    prec_hist.append(precision)
    recall_hist.append(recall)
    f1_hist.append(f1)
    epoch_time_hist.append(epoch_time)

    # PRINT SUMMARY
    print(f"Epoch {epoch}/{EPOCHS} -> "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc*100:.2f}%, "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc*100:.2f}%, "
          f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}, "
          f"Time: {epoch_time:.2f}s")

    scheduler.step(val_acc)

    # SAVE BEST MODEL
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({'model_state': model.state_dict(), 'class_names': class_names}, MODEL_SAVE)
        print(f"Saved best model (val_acc={best_val_acc*100:.2f}%) -> {MODEL_SAVE}")

total_end = time.time()
total_time = total_end - total_start

# ============================
# AVERAGE METRICS
# ============================
print("\n========== AVERAGE METRICS ==========")
print(f"Avg Train Loss: {np.mean(train_loss_hist):.4f}")
print(f"Avg Val Loss: {np.mean(val_loss_hist):.4f}")
print(f"Avg Train Accuracy: {np.mean(train_acc_hist)*100:.2f}%")
print(f"Avg Val Accuracy: {np.mean(val_acc_hist)*100:.2f}%")
print(f"Avg Precision: {np.mean(prec_hist):.4f}")
print(f"Avg Recall: {np.mean(recall_hist):.4f}")
print(f"Avg F1-score: {np.mean(f1_hist):.4f}")
print(f"Avg Time/Epoch: {np.mean(epoch_time_hist):.2f}s")
print(f"Total Training Time: {total_time:.2f}s ({total_time/60:.2f} min)")
print("======================================")

print("\nTraining complete. Best validation accuracy:", best_val_acc)

Classes: ['grade1', 'grade2', 'grade3']
Train images: 480 Val images: 120
Device: cpu


Epoch 1/100 -> Train Loss: 1.0496, Train Acc: 43.96%, Val Loss: 0.9884, Val Acc: 50.00%, Precision: 0.5095, Recall: 0.5000, F1: 0.4866, Time: 149.40s
Saved best model (val_acc=50.00%) -> efficientnet_b0_cpu.pth


Epoch 2/100 -> Train Loss: 0.8758, Train Acc: 65.83%, Val Loss: 0.8875, Val Acc: 62.50%, Precision: 0.6239, Recall: 0.6250, F1: 0.6143, Time: 147.55s
Saved best model (val_acc=62.50%) -> efficientnet_b0_cpu.pth


Epoch 3/100 -> Train Loss: 0.7254, Train Acc: 71.88%, Val Loss: 0.8265, Val Acc: 65.83%, Precision: 0.6558, Recall: 0.6583, F1: 0.6562, Time: 146.38s
Saved best model (val_acc=65.83%) -> efficientnet_b0_cpu.pth


Epoch 4/100 -> Train Loss: 0.6172, Train Acc: 74.17%, Val Loss: 0.7736, Val Acc: 62.50%, Precision: 0.6352, Recall: 0.6250, F1: 0.6268, Time: 146.39s


Epoch 5/100 -> Train Loss: 0.4953, Train Acc: 82.71%, Val Loss: 0.7508, Val Acc: 66.67%, Precision: 0.6693, Recall: 0.6667, F1: 0.6675, Time: 146.40s
Saved best model (val_acc=66.67%) -> efficientnet_b0_cpu.pth


... (91 outputs left out)

Epoch 97/100 -> Train Loss: 0.1581, Train Acc: 93.54%, Val Loss: 1.0045, Val Acc: 63.33%, Precision: 0.6459, Recall: 0.6333, F1: 0.6381, Time: 146.65s


Epoch 98/100 -> Train Loss: 0.1857, Train Acc: 92.29%, Val Loss: 0.9819, Val Acc: 63.33%, Precision: 0.6480, Recall: 0.6333, F1: 0.6380, Time: 147.20s


Epoch 99/100 -> Train Loss: 0.1821, Train Acc: 92.08%, Val Loss: 0.9967, Val Acc: 65.00%, Precision: 0.6638, Recall: 0.6500, F1: 0.6515, Time: 148.21s


Epoch 100/100 -> Train Loss: 0.1771, Train Acc: 92.50%, Val Loss: 0.9960, Val Acc: 61.67%, Precision: 0.6388, Recall: 0.6167, F1: 0.6218, Time: 147.43s

========== AVERAGE METRICS ==========
Avg Train Loss: 0.2268
Avg Val Loss: 0.9839
Avg Train Accuracy: 91.15%
Avg Val Accuracy: 62.09%
Avg Precision: 0.6391
Avg Recall: 0.6209
Avg F1-score: 0.6259
Avg Time/Epoch: 146.41s
Total Training Time: 14641.42s (244.02 min)

Training complete. Best validation accuracy: 0.6666666666666666
